# sim-highline quickstart

Load the dataset, pick one relation at one epoch, plot every simulation against the observations, and get the citations.

Units are log10, physical and h-free. Curves are only comparable when their definition columns agree: always look at `x_definition`, `y_definition`, `mass_definition`, `population` and the others before reading an offset as physics. Nothing here is interpolated.

## Install

In [ ]:
# In Colab or a fresh environment (the repository must be public for this to work):
# %pip install git+https://github.com/ShubhanBhatia42/sim-highline
# From a checkout: %pip install -e .   (data is read from data/export)

## Load

In [ ]:
import sim_highline
import matplotlib.pyplot as plt

df = sim_highline.load()      # tidy table: one row per plotted point
print(f"dataset v{df.attrs['version']}: {df.record_id.nunique()} records, {len(df)} points")
df.groupby('relation').source.nunique().sort_values()

## One relation at one epoch

`nearest_epoch` keeps, for every simulation, the record whose published epoch is closest to `z` (within the same tolerance the website uses) and drops simulations with no epoch there.

In [ ]:
sel = sim_highline.select(df, relation='gsmf', z=8)
one = sim_highline.nearest_epoch(sel, z=8)
one.groupby(['source', 'run', 'kind']).z.first().sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for (src, run, kind, rid), g in one.groupby(['source', 'run', 'kind', 'record_id']):
    g = g.sort_values('point_index')
    ax.plot(g.x, g.y, '--' if kind == 'observation' else '-', label=f'{src} ({run})' if kind == 'simulation' else src, lw=1.5)
ax.set_xlabel('log10 stellar mass [Msun]'); ax.set_ylabel('log10 phi [cMpc^-3 dex^-1]')
ax.legend(fontsize=7, ncol=2); plt.show()

## Check that two curves can be compared

Every record carries its definitions. Here, the mass definition and IMF of each curve in the plot above:

In [ ]:
one.drop_duplicates('record_id')[['source', 'tier', 'rankable', 'mass_definition', 'imf', 'population']].sort_values('source')

## Only evidence that can be ranked

`rankable=True` excludes digitized figures with ambiguous definitions and zoom samples.

In [ ]:
ranked = sim_highline.select(df, relation='sfms', z=(0, 0.1), rankable=True)
ranked.groupby(['source', 'tier']).record_id.nunique()

## Cite what you use

The dataset is cited together with the papers of the records you actually use.

In [ ]:
print(sim_highline.bibtex(one))

Dataset citation: see `CITATION.cff` (Bhatia, S., sim-highline, v0.10.0). Code MIT, data CC BY 4.0.